# Estimación calificación CAMEL

Para obtener la calificación total (usando todos los indicadores previamente calculados) se tienen los siguientes pasos:

1. Descarga y lectura de los datos (Resultado de calificación individual de los indicadores).
2. Extracción y valoración del PCA.
3. Creación de los rangos (percentiles para las variables).
4. Calificación modelo CAMEL - general.
5. Calificación modelo CAMEL - sectorizado

## 1. Extracción de datos

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


In [2]:
data_dir = Path().resolve().parent / "data" / "camel"
filepath = data_dir / "Registros_CAMEL_sin_solvencia_categorizados.csv"
datos=pd.read_csv(filepath)
datos.head()

,ID_registro,ano,mes,ID_indicador,ID_cooperativa,valor,categoria
0,1,2022,1,Quebranto Patrimonial,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,1.265719,Micro 1
1,3,2022,1,Relación entre Aportes sociales mínimos no red...,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,0.431216,Micro 1
2,4,2022,1,Relación entre el Capital Institucional y el A...,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,0.067410,Micro 1
3,5,2022,2,Quebranto Patrimonial,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,1.270411,Micro 1
4,7,2022,2,Relación entre Aportes sociales mínimos no red...,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,0.430248,Micro 1


In [3]:
# en la columna "valor" cambiar nulos por o
datos["valor"] = datos["valor"].fillna(0)
#cuando el valor de la columna "valor" sea ' ', cambiarlo por 0
datos["valor"] = datos["valor"].replace(' ', 0)
#cambiar el tipo de dato de la columna "valor" a float
datos["valor"] = datos["valor"].astype(float)
#cambia top 4
datos.loc[datos["categoria"] == "Top 4", "categoria"] = "Megas"
#'Micro 1', 'Medianas', 'Megas', 'Pequeñas', 'Grandes', 'Micro 2','Top 4'

In [4]:
datos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 133284 entries, 0 to 133283
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   ID_registro     133284 non-null  int64  
 1   ano             133284 non-null  int64  
 2   mes             133284 non-null  int64  
 3   ID_indicador    133284 non-null  object 
 4   ID_cooperativa  133284 non-null  object 
 5   valor           133284 non-null  float64
 6   categoria       133284 non-null  object 
dtypes: float64(1), int64(3), object(3)
memory usage: 7.1+ MB


## 1. Agregación transversal previa

In [50]:
def obtener_matriz_promedios(
    df: pd.DataFrame,
    col_coop: str = "ID_cooperativa",
    col_indicador: str = "ID_indicador",
    col_valor: str = "valor",
) -> pd.DataFrame:
  """Pivotea los datos longitudinales para obtener la matriz estática de
  corte transversal con el valor medio de cada indicador por cooperativa.
  """
  df_temp = df.copy()

  # Limpieza de caracteres y conversión numérica
  df_temp[col_valor] = pd.to_numeric(
      df_temp[col_valor]
      .astype(str)
      .str.replace(",", "", regex=False)
      .str.strip(),
      errors="coerce",
  )

  df_pivot = df_temp.pivot_table(
      index=col_coop, columns=col_indicador, values=col_valor, aggfunc="mean"
  )
  return df_pivot

In [51]:
df_pivot = obtener_matriz_promedios(datos)
df_pivot.head()

ID_indicador,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total
ID_cooperativa,,,,,,,,,,,,,,,,
AVANCOP COOPERATIVA DE AHORRO Y CREDITO,0.875762,1.741230,0.088723,0.346254,0.760854,0.073607,0.452114,0.074736,0.025034,0.102664,0.028257,0.117220,0.262012,1.249912,0.463777,0.072834
CAJA COOPERATIVA CREDICOOP,0.867119,1.746639,0.077716,0.442084,0.770383,-0.057296,1.403776,0.115678,0.123490,-0.092906,0.039640,-0.083857,-0.169038,1.343885,0.945418,0.083444
CAJA COOPERATIVA PETROLERA,0.879444,1.491286,0.080146,0.563161,0.610251,-0.005344,0.380802,0.096615,0.106907,0.005386,0.101211,0.021954,0.064965,1.363090,0.826293,0.078041
CASA NACIONAL DEL PROFESOR,0.897483,1.340547,0.024667,0.536206,0.485378,0.048693,0.174658,0.037486,0.003475,0.034791,0.078702,0.031065,0.111820,1.148259,0.671754,0.028217
CESCA COOPERATIVA DE AHORRO Y CREDITO,0.735429,2.052872,0.055462,0.500876,0.792410,0.178755,1.127639,0.076970,0.038800,0.218761,0.000000,0.192636,0.325967,1.176594,0.202325,0.045620


In [52]:
df_pivot.describe()

ID_indicador,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total
count,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000
mean,0.675256,1.469195,0.050338,0.330018,0.563382,0.044041,0.790182,0.064556,0.033598,0.090349,0.036854,0.134058,1.162885,1.267179,0.514267,0.074187
std,0.315849,1.116213,0.036227,0.198695,0.275461,0.125881,2.098235,0.049119,0.050898,0.142900,0.063057,0.261625,6.211047,0.722727,0.299710,0.057043
min,0.000000,0.000000,0.000000,-0.056947,0.000000,-0.558242,0.000000,0.000000,0.000000,-0.574162,0.000000,-0.362890,-0.537200,0.000000,0.000000,0.000000
25%,0.534614,1.020330,0.025584,0.185217,0.459805,0.000000,0.283008,0.029533,0.003781,0.000000,0.000000,0.001448,0.013818,1.067501,0.306775,0.033750
50%,0.821298,1.321915,0.043872,0.368652,0.635468,0.032564,0.450879,0.060881,0.016578,0.075350,0.004853,0.086135,0.245317,1.316822,0.564003,0.067355
75%,0.879458,1.693774,0.070845,0.478116,0.770097,0.118467,0.742770,0.088079,0.043412,0.153822,0.036402,0.201866,0.511520,1.601885,0.752572,0.101965
max,0.985357,6.612775,0.166095,0.748502,0.968679,0.388762,26.500825,0.283015,0.427148,0.540305,0.323367,2.651342,57.539497,5.178508,1.000000,0.316550


## 2. Obtener PCA

In [53]:
def pesos_pca_matriz(df_pivot, n_componentes=3):
  """Calcula los pesos normalizados por PCA sobre la matriz transversal de cooperativas."""
  imputer = SimpleImputer(strategy="mean")
  X_imputed = imputer.fit_transform(df_pivot)

  scaler = StandardScaler()
  X_scaled = scaler.fit_transform(X_imputed)

  n_comp = min(n_componentes, df_pivot.shape[1], df_pivot.shape[0])
  pca = PCA(n_components=n_comp)
  pca.fit(X_scaled)
  X_pca = pca.transform(X_scaled)

  loadings = pd.DataFrame(pca.components_.T, index=df_pivot.columns)
  pesos = (loadings**2).mean(axis=1)
  pesos = pesos / pesos.sum()

  pesos_ponderados = np.sum(
      (loadings.iloc[:, :n_componentes]**2)*pca.singular_values_[:n_componentes], 
      axis=1
      )
  pesos_ponderados = (pesos_ponderados / pesos_ponderados.sum())

  # Varianza acumulada
  varianza_acumulada = pd.DataFrame(
  np.cumsum(pca.explained_variance_ratio_), 
  index=[f"PC{i+1}" for i in range(X_pca.shape[1])], 
  columns=["Varianza acumuluda explicada"]
  )

  return pesos, pesos_ponderados, varianza_acumulada

In [54]:
pesos, pesos_ponderados, varianza_acumulada = pesos_pca_matriz(df_pivot, n_componentes=6)

pesos.sort_values(ascending=False)

ID_indicador
Indicador de Riesgo de Liquidez - IRL                                                     0.141980
Indicador de calidad por riesgo con castigos                                              0.113808
Indicador de relación entre las obligaciones financieras y el pasivo total                0.109550
Indicador de rentabilidad sobre recursos propios - ROE                                    0.095248
Indicador de rentabilidad sobre el capital invertido - ROIC                               0.071793
Estructura de Balance                                                                     0.068373
Indicador de margen neto                                                                  0.059192
Indicador de Margen Operacional                                                           0.057676
Indicador de calidad por riesgo                                                           0.054197
Indicador de Cobertura de la Cartera Total en Riesgo                                      0.0429

In [55]:
pesos_ponderados.sort_values(ascending=False)

ID_indicador
Indicador de Riesgo de Liquidez - IRL                                                     0.104647
Indicador de relación entre las obligaciones financieras y el pasivo total                0.087042
Indicador de rentabilidad sobre recursos propios - ROE                                    0.086528
Indicador de calidad por riesgo con castigos                                              0.085794
Indicador de rentabilidad sobre el capital invertido - ROIC                               0.075651
Indicador de margen neto                                                                  0.066536
Indicador de Margen Operacional                                                           0.064553
Estructura de Balance                                                                     0.060837
Indicador de calidad por riesgo                                                           0.057843
Indicador de Cobertura de la Cartera Total en Riesgo                                      0.0509

In [56]:
varianza_acumulada

,Varianza acumuluda explicada
PC1,0.377471
PC2,0.568265
PC3,0.670973
PC4,0.737128
PC5,0.790822
PC6,0.830710


## 3. Cálculo Rangos

In [57]:
INDICADORES_INVERSOS = [
    "Indicador de calidad por riesgo",
    "Indicador de calidad por riesgo con castigos",
    "Indicador de Cobertura de la Cartera Total en Riesgo",
    "Indicador de relación entre las obligaciones financieras y el pasivo total",
    "Indicador de Margen Financiero de Operación",
    "Indicador de Margen Operacional",
]

# Ajuste exacto al nombre en el CSV
INDICADOR_IRL = "Indicador de Riesgo de Liquidez - IRL"
INDICADOR_SOLVENCIA = "Relación Solvencia"
PERCENTILES = [20, 40, 60, 80]

In [64]:
def calcular_percentiles_matriz(df_pivot: pd.DataFrame) -> dict:
  """Calcula los quintiles empíricos (P20, P40, P60, P80) por indicador

  directamente sobre el conjunto transversal de cooperativas.
  """
  cortes_dict = {}
  primer_corte = PERCENTILES[0]  # 20
  cortes_restantes = PERCENTILES[1:]  # [40, 60, 80]

  for indicador in df_pivot.columns:
    valores = df_pivot[indicador].dropna().copy()

    # Ajuste por exceso de masa en cero (>= 10%)
    porcentaje_ceros = (valores == 0).mean()
    if porcentaje_ceros >= 0.10:
      valores_sin_cero = valores[valores > 0]
      p = {primer_corte: 0.0}

      if len(valores_sin_cero) > 0:
        for q in cortes_restantes:
          q_ajustado = (q - primer_corte) / (100 - primer_corte)
          p[q] = valores_sin_cero.quantile(q_ajustado)
      else:
        for q in cortes_restantes:
          p[q] = 0.0
    else:
      p = {q: valores.quantile(q / 100) for q in PERCENTILES}

    # Anclaje de solvencia al piso normativo
    if indicador == INDICADOR_SOLVENCIA:
      p[primer_corte] = 9.0
      for q in cortes_restantes:
        p[q] = max(p[q], p[primer_corte])

    cortes_dict[indicador] = p

  return cortes_dict

def asignar_calificacion(valor: float, cortes: dict, inverso: bool = False) -> int:
  """Asigna la calificación ordinal en escala 1 a 5 respetando

  la monotonicidad del riesgo para variables directas e inversas.
  """
  p20 = cortes[20]
  p40 = cortes[40]
  p60 = cortes[60]
  p80 = cortes[80]

  if not inverso:
    if valor <= p20:
      return 1
    elif valor <= p40:
      return 2
    elif valor <= p60:
      return 3
    elif valor <= p80:
      return 4
    else:
      return 5
  else:
    # Indicadores inversos: menor valor representa mejor desempeño
    if valor > p80:
      return 1
    elif valor > p60:
      return 2
    elif valor > p40:
      return 3
    elif valor > p20:
      return 4
    else:  # valor <= p20 (incluye valores en 0.0 en el quintil superior)
      return 5


def calificar_matriz(df_pivot: pd.DataFrame, cortes_dict: dict) -> pd.DataFrame:
  """Evalúa cada razón financiera promedio de la entidad,

  asignando un puntaje discreto entero entre 1 y 5.
  """
  df_calif = pd.DataFrame(index=df_pivot.index, columns=df_pivot.columns)

  for col in df_pivot.columns:
    inv = col in INDICADORES_INVERSOS
    cortes = cortes_dict[col]

    df_calif[col] = df_pivot[col].apply(
        lambda val: np.nan
        if pd.isna(val)
        else asignar_calificacion(val, cortes, inverso=inv)
    )

  return df_calif

In [65]:
df_calificaciones = calificar_matriz(df_pivot, calcular_percentiles_matriz(df_pivot))
df_calificaciones.head()

ID_indicador,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total
ID_cooperativa,,,,,,,,,,,,,,,,
AVANCOP COOPERATIVA DE AHORRO Y CREDITO,4,4,1,3,2,3,3,2,2,3,2,3,3,3,3,3
CAJA COOPERATIVA CREDICOOP,4,4,1,4,2,5,5,1,1,1,2,1,1,3,5,4
CAJA COOPERATIVA PETROLERA,4,4,1,5,3,5,3,1,1,2,1,2,2,4,5,4
CASA NACIONAL DEL PROFESOR,5,3,4,5,4,3,2,4,4,2,1,2,2,2,4,2
CESCA COOPERATIVA DE AHORRO Y CREDITO,2,5,2,5,2,1,5,2,2,5,5,4,3,2,2,2


In [66]:
df_calificaciones.describe()

ID_indicador,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total
count,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000,174.000000
mean,3.189655,3.189655,2.810345,3.155172,2.810345,3.419540,3.183908,2.810345,2.810345,2.821839,3.189655,2.896552,2.896552,3.189655,3.189655,3.183908
std,1.331567,1.331567,1.331567,1.357510,1.331567,1.506006,1.338870,1.331567,1.331567,1.469295,1.475702,1.450808,1.450808,1.331567,1.331567,1.338870
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
50%,3.000000,3.000000,3.000000,3.000000,3.000000,4.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
75%,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


In [67]:
df_calificaciones['Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo'].value_counts()

Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo
5    38
2    38
3    37
4    37
1    24
Name: count, dtype: int64

In [68]:
df_calificaciones['Indicador de Cobertura de la Cartera Total en Riesgo'].value_counts()

Indicador de Cobertura de la Cartera Total en Riesgo
4    39
1    38
2    38
3    38
5    21
Name: count, dtype: int64

In [69]:
df_calificaciones['Indicador de Margen Financiero de Operación'].value_counts()

Indicador de Margen Financiero de Operación
4    39
2    38
3    38
1    38
5    21
Name: count, dtype: int64

In [70]:
df_calificaciones['Indicador de calidad por riesgo'].value_counts()

Indicador de calidad por riesgo
4    39
2    38
1    38
3    38
5    21
Name: count, dtype: int64

In [71]:
df_calificaciones['Indicador de calidad por riesgo con castigos'].value_counts()

Indicador de calidad por riesgo con castigos
4    39
2    38
1    38
3    38
5    21
Name: count, dtype: int64

In [72]:
df_calificaciones['Indicador de relación entre las obligaciones financieras y el pasivo total'].value_counts()

Indicador de relación entre las obligaciones financieras y el pasivo total
5    48
4    32
1    32
2    31
3    31
Name: count, dtype: int64

## 4. Cálculo del CAMEL score

In [73]:
def calcular_camel(df_calif: pd.DataFrame, pesos: pd.Series) -> pd.DataFrame:
  """Calcula la puntuación agregada ponderada por entidad.

  No requiere renormalizar pesos puesto que la serie ya suma 1.0.
  """
  cols_comunes = [c for c in pesos.index if c in df_calif.columns]
  score = (df_calif[cols_comunes].fillna(0) * pesos[cols_comunes]).sum(axis=1)

  return pd.DataFrame(
      {"ID_cooperativa": df_calif.index, "CAMEL_score": score.values}
  )

In [74]:
df_resultado = calcular_camel(df_calificaciones, pesos_ponderados)
df_resultado.head()

,ID_cooperativa,CAMEL_score
0,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,2.728621
1,CAJA COOPERATIVA CREDICOOP,2.643413
2,CAJA COOPERATIVA PETROLERA,2.699722
3,CASA NACIONAL DEL PROFESOR,2.874545
4,CESCA COOPERATIVA DE AHORRO Y CREDITO,3.248975


In [75]:
df_resultado['CAMEL_score'].describe()

count    174.000000
mean       3.039011
std        0.451581
min        2.102127
25%        2.601159
50%        3.062673
75%        3.375241
max        4.028841
Name: CAMEL_score, dtype: float64

In [76]:
df_resultado_sin_ponderar = calcular_camel(df_calificaciones, pesos)
df_resultado_sin_ponderar.head()

,ID_cooperativa,CAMEL_score
0,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,2.709385
1,CAJA COOPERATIVA CREDICOOP,2.624778
2,CAJA COOPERATIVA PETROLERA,2.539826
3,CASA NACIONAL DEL PROFESOR,2.748705
4,CESCA COOPERATIVA DE AHORRO Y CREDITO,3.398776


In [77]:
df_resultado_sin_ponderar['CAMEL_score'].describe()

count    174.000000
mean       3.039738
std        0.492869
min        1.932576
25%        2.623508
50%        2.992158
75%        3.464409
max        4.116937
Name: CAMEL_score, dtype: float64

In [78]:
df_indicadores_camel = pd.merge(
    df_calificaciones,
    df_resultado,
    on="ID_cooperativa",
    how="left"
)

df_indicadores_camel.rename(columns={"CAMEL_score": "CAMEL_score_ponderado"}, inplace=True)
df_indicadores_camel

,ID_cooperativa,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total,CAMEL_score_ponderado
0,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,4,4,1,3,2,3,3,2,2,3,2,3,3,3,3,3,2.728621
1,CAJA COOPERATIVA CREDICOOP,4,4,1,4,2,5,5,1,1,1,2,1,1,3,5,4,2.643413
2,CAJA COOPERATIVA PETROLERA,4,4,1,5,3,5,3,1,1,2,1,2,2,4,5,4,2.699722
3,CASA NACIONAL DEL PROFESOR,5,3,4,5,4,3,2,4,4,2,1,2,2,2,4,2,2.874545
4,CESCA COOPERATIVA DE AHORRO Y CREDITO,2,5,2,5,2,1,5,2,2,5,5,4,3,2,2,2,3.248975
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURS...,5,5,2,2,1,3,5,3,4,5,4,5,4,4,2,5,3.856030
170,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,5,4,4,2,2,2,3,3,3,4,1,5,5,2,4,2,3.240754
171,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORR...,2,4,1,5,1,1,5,1,1,5,3,5,5,5,3,5,3.307896
172,ORGANIZACION COOPERATIVA DE AHORRO Y CREDITO N...,3,4,1,5,2,1,4,2,2,5,5,4,4,5,3,5,3.451269


In [79]:
df_indicadores_camel = pd.merge(
    df_indicadores_camel,
    df_resultado_sin_ponderar,
    on="ID_cooperativa",
    how="left"
)

df_indicadores_camel.rename(columns={"CAMEL_score": "CAMEL_score_sin_ponderar"}, inplace=True)
df_indicadores_camel

,ID_cooperativa,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total,CAMEL_score_ponderado,CAMEL_score_sin_ponderar
0,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,4,4,1,3,2,3,3,2,2,3,2,3,3,3,3,3,2.728621,2.709385
1,CAJA COOPERATIVA CREDICOOP,4,4,1,4,2,5,5,1,1,1,2,1,1,3,5,4,2.643413,2.624778
2,CAJA COOPERATIVA PETROLERA,4,4,1,5,3,5,3,1,1,2,1,2,2,4,5,4,2.699722,2.539826
3,CASA NACIONAL DEL PROFESOR,5,3,4,5,4,3,2,4,4,2,1,2,2,2,4,2,2.874545,2.748705
4,CESCA COOPERATIVA DE AHORRO Y CREDITO,2,5,2,5,2,1,5,2,2,5,5,4,3,2,2,2,3.248975,3.398776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURS...,5,5,2,2,1,3,5,3,4,5,4,5,4,4,2,5,3.856030,3.992724
170,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,5,4,4,2,2,2,3,3,3,4,1,5,5,2,4,2,3.240754,3.217163
171,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORR...,2,4,1,5,1,1,5,1,1,5,3,5,5,5,3,5,3.307896,3.352650
172,ORGANIZACION COOPERATIVA DE AHORRO Y CREDITO N...,3,4,1,5,2,1,4,2,2,5,5,4,4,5,3,5,3.451269,3.490593


In [80]:
df_indicadores_camel = pd.merge(
    df_indicadores_camel,
    datos[["ID_cooperativa", "categoria"]].drop_duplicates(),
    on="ID_cooperativa",
    how="left"
)

df_indicadores_camel

,ID_cooperativa,Activo Productivo,Estructura de Balance,Indicador de Cobertura de la Cartera Total en Riesgo,Indicador de Cobertura individual de la cartera improductiva para la cartera en Riesgo,Indicador de Margen Financiero de Operación,Indicador de Margen Operacional,Indicador de Riesgo de Liquidez - IRL,Indicador de calidad por riesgo,Indicador de calidad por riesgo con castigos,Indicador de margen neto,Indicador de relación entre las obligaciones financieras y el pasivo total,Indicador de rentabilidad sobre el capital invertido - ROIC,Indicador de rentabilidad sobre recursos propios - ROE,Quebranto Patrimonial,Relación entre Aportes sociales mínimos no reducibles y Capital Social,Relación entre el Capital Institucional y el Activo Total,CAMEL_score_ponderado,CAMEL_score_sin_ponderar,categoria
0,AVANCOP COOPERATIVA DE AHORRO Y CREDITO,4,4,1,3,2,3,3,2,2,3,2,3,3,3,3,3,2.728621,2.709385,Micro 1
1,CAJA COOPERATIVA CREDICOOP,4,4,1,4,2,5,5,1,1,1,2,1,1,3,5,4,2.643413,2.624778,Medianas
2,CAJA COOPERATIVA PETROLERA,4,4,1,5,3,5,3,1,1,2,1,2,2,4,5,4,2.699722,2.539826,Megas
3,CASA NACIONAL DEL PROFESOR,5,3,4,5,4,3,2,4,4,2,1,2,2,2,4,2,2.874545,2.748705,Megas
4,CESCA COOPERATIVA DE AHORRO Y CREDITO,2,5,2,5,2,1,5,2,2,5,5,4,3,2,2,2,3.248975,3.398776,Pequeñas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,GRAN COOPERATIVA DE ENERGIA ELECTRICA Y RECURS...,5,5,2,2,1,3,5,3,4,5,4,5,4,4,2,5,3.856030,3.992724,Micro 1
170,LA COOPERATIVA DE AHORRO Y CREDITO SUCREDITO,5,4,4,2,2,2,3,3,3,4,1,5,5,2,4,2,3.240754,3.217163,Micro 1
171,MICROEMPRESAS DE COLOMBIA COOPERATIVA DE AHORR...,2,4,1,5,1,1,5,1,1,5,3,5,5,5,3,5,3.307896,3.352650,Grandes
172,ORGANIZACION COOPERATIVA DE AHORRO Y CREDITO N...,3,4,1,5,2,1,4,2,2,5,5,4,4,5,3,5,3.451269,3.490593,Medianas


In [81]:
df_indicadores_camel.to_csv(data_dir / "Calificaciones_CAMEL.csv", index=False)